# Task 3: Assessing performance degradation under data shifts
A) We'll create multiple test sets from the original test set with increasingly strong data shifts
B) We'll assess performance degradation of the model under these data shifts

In [1]:
%load_ext autoreload
%autoreload 2

### Required package imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics import balanced_accuracy_score

import torch

from resnet_hearthealth_monitor.config import DEVICE
from resnet_hearthealth_monitor.data_processing.augmentations import generate_holdout_using_augmentations
from resnet_hearthealth_monitor.modeling.predict import testset_prediction
from resnet_hearthealth_monitor.modeling.resnet import ResNet125
from resnet_hearthealth_monitor.data_processing.pytorch_datasets import augmentable_dataset


2025-07-29 15:35:14.513 | INFO     | lightweight_resnet_heart_monitor.config:<module>:12 - PROJ_ROOT path is: /home/anthony/home/github/lightweight_resnet_heart_monitor
/home/anthony/.virtualenvs/lightweight_resnet_heart_monitor/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-07-29 15:35:14.564 | INFO     | lightweight_resnet_heart_monitor.config:<module>:41 - DEVICE is set to: cuda


In [3]:
device = DEVICE
if device == 'cpu':
    print("WARNING: CUDA not available. Using CPU for training, which may be very slow.")

# A) Creating multiple additional test sets from the original test set to assess performance degradation under data shifts

### We will create multiple holdout sets using increasingly strong shifts

In [4]:
# Define base augmentation parameters
initial_shift_strengths = {
    "wander_factor": 0.05,   # base amplitude for baseline wander
    "frequency": 0.5,        # frequency for baseline wander (kept constant)
    "noise_factor": 0.05,    # base standard deviation for Gaussian noise
    "amplify_factor": 1.2    # base amplitude scaling factor
}

# Define a list of multipliers to increase augmentation strength
shift_multipliers = [1.0, 1.5, 2.0, 2.5]

test_set = pd.read_csv('../data/processed/mitbih_test_cropped.csv', header=None)  

collected_test_sets_after_shifting = []
for multiplier in shift_multipliers:
    X_test = test_set.iloc[:, :-1]
    y_test = test_set.iloc[:, -1]

    # Update the augmentation parameters based on the current multiplier
    shift_config = {
         "wander_factor": initial_shift_strengths["wander_factor"] * multiplier,
         "frequency": initial_shift_strengths["frequency"],  # keeping frequency constant
         "noise_factor": initial_shift_strengths["noise_factor"] * multiplier,
         "amplify_factor": initial_shift_strengths["amplify_factor"] * multiplier
    }
    
    # Apply augmentations to each test sample using your augmentation function.
    # Here, `generate_holdout_using_augmentations` should accept a sample (row) and shift_config.
    X_test_shifted = X_test.apply(lambda row: generate_holdout_using_augmentations(row, shift_config), axis=1)
    
    # Convert the Series of arrays to a 2D NumPy array.
    X_test_shifted = pd.DataFrame(np.stack(X_test_shifted.values))
    df_labels = y_test.reset_index(drop=True)
    # Merge the shifted features with the true labels
    test_set_shifted = pd.concat([X_test_shifted, df_labels], axis=1)
    collected_test_sets_after_shifting.append(test_set_shifted)
    

# B) Assessing performance degradation of the model under data shifts

As seen below, the model performs increasingly poorly as the data shifts get stronger. 

The model's precision for minority classes (classes 1-4) remains a bit more robust to the shifts, relative to the model's recall abiltiies for minority classes. To illustrate, when we move from shift strength of 1 to 2.5, macro recall drops 59 points from .89 to .30 while precision only drops 13 points from .77 to .64. 

In contrast, the majority class, class 0, has seen a greater drop in precision from .99 to .86, while recall has increased a little from .96 to .99.

Together this indicates that when the model makes a prediction of one of the minority classes, it remains relatively likely to be true. However, the greatest impact to model performance has come from an increase in mistakingly categorizing one of the minority classes as the healthy timeseries majority class. This explains the pattern of reducing precision in the majority class and reducing recall in the minority classes. 

In [5]:
torch_model_class = ResNet125()
torch_dataset_class = augmentable_dataset

for i in range(0, len(shift_multipliers), 1):
    multiplier = shift_multipliers[i]
    print(f"\n=== Shift Multiplier: {multiplier} ===")

    test_set_shifted = collected_test_sets_after_shifting[i]

    # Run your model inference on the shifted test data.
    # The function testset_prediction should accept the test set tuple and return a DataFrame
    # with columns 'Prediction' and 'Label'.
    testset_predictions_and_labels_df = testset_prediction(test_set_shifted, torch_model_class, torch_dataset_class, 
                                                           device, save_and_display_results=False)
    
    # Evaluate the predictions
    predictions = testset_predictions_and_labels_df['Prediction']
    labels = testset_predictions_and_labels_df['Label']
    print(f"classification report: \n {classification_report(labels, predictions, zero_division=np.nan)}")


=== Shift Multiplier: 1.0 ===
2025-07-29 15:37:14.328 | SUCCESS  | lightweight_resnet_heart_monitor.modeling.predict:testset_prediction:80 - test-set predictions generated
classification report: 
               precision    recall  f1-score   support

         0.0       0.99      0.96      0.98     18118
         1.0       0.56      0.77      0.65       556
         2.0       0.85      0.93      0.89      1448
         3.0       0.48      0.83      0.61       162
         4.0       0.98      0.97      0.98      1608

    accuracy                           0.96     21892
   macro avg       0.77      0.89      0.82     21892
weighted avg       0.96      0.96      0.96     21892


=== Shift Multiplier: 1.5 ===
2025-07-29 15:37:22.487 | SUCCESS  | lightweight_resnet_heart_monitor.modeling.predict:testset_prediction:80 - test-set predictions generated
classification report: 
               precision    recall  f1-score   support

         0.0       0.95      0.96      0.96     18118
      